# Data Quality Analysis

Análisis de calidad de datos de la capa Bronze
para identificar problemas de consistencia,
valores nulos, duplicados y registros inválidos
antes de construir la capa Silver.

## Función general de análisis de calidad

Se crea una función reutilizable llamada `analizar_calidad()`
que permitirá ejecutar automáticamente múltiples validaciones
sobre cada tabla de la capa Bronze.

La función analiza:

- Total de registros
- Esquema de datos
- Valores nulos
- Porcentaje de nulos
- Strings vacíos
- Registros duplicados
- Estadísticas de columnas numéricas
- Valores negativos
- Cardinalidad
- Columnas constantes
- Muestra de datos

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

def analizar_calidad(tabla, columna_id=None):

    print(f"\n{'='*80}")
    print(f"ANÁLISIS DE CALIDAD: {tabla}")
    print(f"{'='*80}")

    df = spark.table(tabla)

    total_registros = df.count()

    print(f"\nTotal de registros: {total_registros}")

    # ==========================================
    # ESQUEMA
    # ==========================================

    print("\nESQUEMA DE DATOS")
    df.printSchema()

    # ==========================================
    # NULOS Y PORCENTAJES
    # ==========================================

    print("\nANÁLISIS DE VALORES NULOS")

    nulls = df.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in df.columns
    ])

    display(nulls)

    porcentajes = df.select([
        (
            (count(when(col(c).isNull(), c)) / total_registros) * 100
        ).alias(c)
        for c in df.columns
    ])

    print("\nPORCENTAJE DE NULOS (%)")
    display(porcentajes)

    # ==========================================
    # STRINGS VACÍOS
    # ==========================================

    print("\nSTRINGS VACÍOS")

    columnas_string = [
        field.name
        for field in df.schema.fields
        if isinstance(field.dataType, StringType)
    ]

    if columnas_string:

        vacios = df.select([
            count(when(trim(col(c)) == "", c)).alias(c)
            for c in columnas_string
        ])

        display(vacios)

    # ==========================================
    # DUPLICADOS
    # ==========================================

    if columna_id:

        print(f"\nANÁLISIS DE DUPLICADOS EN {columna_id}")

        duplicados = (
            df.groupBy(columna_id)
            .count()
            .filter(col("count") > 1)
        )

        total_duplicados = duplicados.count()

        print(f"Total duplicados encontrados: {total_duplicados}")

        display(duplicados)

    # ==========================================
    # ESTADÍSTICAS NUMÉRICAS
    # ==========================================

    print("\nESTADÍSTICAS DE COLUMNAS NUMÉRICAS")

    columnas_numericas = [
        field.name
        for field in df.schema.fields
        if isinstance(
            field.dataType,
            (
                IntegerType,
                LongType,
                DoubleType,
                FloatType,
                DecimalType
            )
        )
    ]

    if columnas_numericas:

        display(
            df.select(columnas_numericas).describe()
        )

    # ==========================================
    # VALORES NEGATIVOS
    # ==========================================

    print("\nANÁLISIS DE VALORES NEGATIVOS")

    for c in columnas_numericas:

        negativos = df.filter(col(c) < 0).count()

        if negativos > 0:

            print(f"{c}: {negativos} valores negativos")

    # ==========================================
    # CARDINALIDAD
    # ==========================================

    print("\nCARDINALIDAD DE COLUMNAS")

    cardinalidad = []

    for c in df.columns:

        cardinalidad.append(
            (
                c,
                df.select(c).distinct().count()
            )
        )

    cardinalidad_df = spark.createDataFrame(
        cardinalidad,
        ["columna", "valores_unicos"]
    )

    display(cardinalidad_df)

    # ==========================================
    # COLUMNAS CONSTANTES
    # ==========================================

    print("\nCOLUMNAS CONSTANTES")

    for c in df.columns:

        unicos = df.select(c).distinct().count()

        if unicos == 1:

            print(f"{c} tiene un único valor")

    # ==========================================
    # MUESTRA DE DATOS
    # ==========================================

    print("\nMUESTRA DE DATOS")

    display(df.limit(10))

    print(f"\nFINALIZADO: {tabla}")

## Análisis de bronze_bookings

Se analiza la tabla de reservas para identificar problemas de calidad
relacionados con registros de reservas, usuarios, propiedades y fechas.

In [0]:
analizar_calidad(
    "bronze.bronze_bookings",
    "booking_id"
)

## Análisis de bronze_users

Se analiza la tabla de usuarios para validar información de clientes,
campos incompletos, duplicados y posibles inconsistencias.

In [0]:
analizar_calidad(
    "bronze.bronze_users",
    "user_id"
)

## Análisis de bronze_payments

Se analiza la tabla de pagos para detectar valores negativos,
campos nulos y problemas relacionados con transacciones financieras.

In [0]:
analizar_calidad(
    "bronze.bronze_payments",
    "payment_id"
)

## Análisis de bronze_properties

Se analiza la tabla de propiedades para validar información
de alojamientos, ubicaciones y atributos descriptivos.

In [0]:
analizar_calidad(
    "bronze.bronze_properties",
    "property_id"
)

## Análisis de bronze_reviews

Se analiza la tabla de reseñas para identificar posibles
problemas en calificaciones, comentarios y referencias de usuarios.

In [0]:
analizar_calidad(
    "bronze.bronze_reviews",
    "review_id"
)

# Conclusiones del análisis de calidad

El análisis permitió identificar posibles problemas de calidad
presentes en la capa Bronze del proyecto Wanderbricks.

Los hallazgos obtenidos servirán como base para implementar
las reglas de limpieza, transformación y estandarización
durante la construcción de la capa Silver.

Entre los principales aspectos evaluados se encuentran:

- Valores nulos
- Registros duplicados
- Inconsistencias numéricas
- Valores negativos
- Strings vacíos
- Cardinalidad de columnas
- Estructura de datos

La siguiente fase del proyecto consistirá en construir
la capa Silver aplicando las transformaciones necesarias
para mejorar la calidad y consistencia de los datos.